In [ ]:
import gc
import torch

# 先清理一下上一个 notebook 可能残留的显存缓存。
gc.collect()
if torch.cuda.is_available():
    torch.cuda.empty_cache()
    torch.cuda.ipc_collect()
    print('GPU cache cleared.')
else:
    print('CUDA not available, skip GPU cache clear.')


# 下一课：MCTS 入门

在开始做简化版 AlphaGo Zero 之前，你非常值得先把 `MCTS` 搞明白。

`MCTS` 的全称是 `Monte Carlo Tree Search`，中文一般叫：

**蒙特卡洛树搜索**

它是 AlphaGo / AlphaGo Zero / AlphaZero 里最关键的组件之一。

这节课你会学到：
- MCTS 到底在做什么
- 它为什么不是普通的暴力搜索
- 它的四个核心步骤：选择、扩展、模拟、回传
- 一个最小可运行的 MCTS 例子


## 1. MCTS 是干什么的

如果你在下棋，一个最朴素的想法是：
- 每个合法动作都试一试
- 往后一直算很多步
- 看最后哪个动作更好

但问题是：
- 棋类分支很多
- 算到很深会非常贵
- 不可能把整棵树都搜完

MCTS 的想法就是：

**把计算预算优先花在“更有希望”的分支上。**

它既不是盲目暴力搜索，也不是只凭直觉乱猜，而是一种“边试边把搜索重点往强分支集中”的方法。


## 2. 为什么叫“蒙特卡洛”

`Monte Carlo` 这个词通常表示：
- 用随机采样近似复杂问题

在 MCTS 里，这体现在：
- 从某个节点往后，不一定精确算到终局
- 可以通过随机模拟（rollout）来估计这个节点大概好不好

所以“蒙特卡洛”对应的是：

**用随机模拟来估计局面的价值。**


## 3. MCTS 的四个核心步骤

每做一次搜索迭代，MCTS 通常会经历四步：

1. `Selection` 选择
   从根节点开始，沿着当前最值得探索的子节点一直往下走。

2. `Expansion` 扩展
   走到一个还没完全展开的节点时，新增一个子节点。

3. `Simulation` 模拟
   从新节点开始，随机或用简单策略往后模拟，直到终局。

4. `Backpropagation` 回传
   把这次模拟结果一路传回祖先节点，更新它们的统计量。

这四步不断重复，根节点下每个动作的统计信息就会越来越靠谱。


## 4. 这节课为什么先用井字棋

直接上围棋会让规则和搜索一起变复杂，所以这节课我们先用 `Tic-Tac-Toe`（井字棋）来讲 MCTS。

原因很简单：
- 状态表示直观
- 动作空间小
- 很适合看清搜索树是怎么长出来的

等你把这节课看懂，再切到围棋时，主线会清晰很多。


In [ ]:
import math
import random
from dataclasses import dataclass, field

random.seed(42)


In [ ]:
class TicTacToe:
    def __init__(self, auto_reset=True):
        if auto_reset:
            self.reset()

    def reset(self):
        self.board = [0] * 9  # 0=空, 1=X, -1=O
        self.current_player = 1
        return self.clone()

    def clone(self):
        new_env = TicTacToe(auto_reset=False)
        new_env.board = self.board[:]
        new_env.current_player = self.current_player
        return new_env

    def legal_actions(self):
        return [i for i, v in enumerate(self.board) if v == 0]

    def step(self, action):
        if self.board[action] != 0:
            raise ValueError('非法动作')
        self.board[action] = self.current_player
        self.current_player *= -1

    def winner(self):
        lines = [
            (0, 1, 2), (3, 4, 5), (6, 7, 8),
            (0, 3, 6), (1, 4, 7), (2, 5, 8),
            (0, 4, 8), (2, 4, 6)
        ]
        for a, b, c in lines:
            s = self.board[a] + self.board[b] + self.board[c]
            if s == 3:
                return 1
            if s == -3:
                return -1
        return 0

    def is_terminal(self):
        return self.winner() != 0 or all(v != 0 for v in self.board)

    def result_for_player(self, player):
        w = self.winner()
        if w == player:
            return 1.0
        if w == 0:
            return 0.0
        return -1.0

    def render(self):
        mapping = {1: 'X', -1: 'O', 0: '.'}
        rows = []
        for i in range(0, 9, 3):
            rows.append(' '.join(mapping[self.board[i + j]] for j in range(3)))
        return '\n'.join(rows)


In [ ]:
env = TicTacToe()
print(env.render())
print('当前玩家:', 'X' if env.current_player == 1 else 'O')
print('合法动作:', env.legal_actions())


## 5. 节点里要记录什么

MCTS 里的每个节点，通常至少会记录：
- 当前局面
- 父节点
- 子节点
- 这个节点访问了多少次 `N`
- 这个节点累计价值是多少 `W`
- 平均价值 `Q = W / N`

这些统计量会在回传阶段不断更新。


In [ ]:
@dataclass
class MCTSNode:
    state: TicTacToe
    parent: 'MCTSNode' = None
    action_taken: int = None
    children: dict = field(default_factory=dict)
    visits: int = 0
    value_sum: float = 0.0
    untried_actions: list = field(default_factory=list)

    def __post_init__(self):
        if not self.untried_actions:
            self.untried_actions = self.state.legal_actions()[:]

    @property
    def q_value(self):
        if self.visits == 0:
            return 0.0
        return self.value_sum / self.visits

    def is_fully_expanded(self):
        return len(self.untried_actions) == 0


## 6. 选择阶段：为什么要用 UCB / UCT

选择阶段最关键的问题是：

- 是继续走当前看起来最强的分支
- 还是试试访问较少、但可能有潜力的分支

MCTS 常用 `UCT` 公式来平衡这两件事：

$UCT = Q + c \sqrt{\frac{\ln N_{parent}}{N_{child}}}$

它同时考虑：
- `Q`：这个子节点目前看起来有多好
- 探索项：这个子节点是不是还访问得太少

所以 UCT 的本质就是：

**在“利用当前最优分支”和“探索未知分支”之间做平衡。**


In [ ]:
def uct_score(parent, child, c=1.4):
    if child.visits == 0:
        return float('inf')
    exploitation = child.q_value
    exploration = c * math.sqrt(math.log(parent.visits) / child.visits)
    return exploitation + exploration


def select_child(node, c=1.4):
    return max(node.children.values(), key=lambda child: uct_score(node, child, c=c))


## 7. 扩展、模拟、回传

接下来三步分别是：

- `Expansion`：把一个还没试过的动作扩出来
- `Simulation`：从新节点开始随机下到终局
- `Backpropagation`：把结果一路传回根节点

这三步加上前面的 Selection，就构成了一次完整的 MCTS 迭代。


In [ ]:
def expand(node):
    action = random.choice(node.untried_actions)
    node.untried_actions.remove(action)

    next_state = node.state.clone()
    next_state.step(action)
    child = MCTSNode(state=next_state, parent=node, action_taken=action)
    node.children[action] = child
    return child


def rollout(state, root_player):
    sim_state = state.clone()
    while not sim_state.is_terminal():
        action = random.choice(sim_state.legal_actions())
        sim_state.step(action)
    return sim_state.result_for_player(root_player)


def backpropagate(node, value):
    current = node
    while current is not None:
        current.visits += 1
        current.value_sum += value
        value = -value  # 对手视角相反
        current = current.parent


## 8. 把 MCTS 四步串起来

下面就是完整的一轮 MCTS 搜索。

根节点当前玩家会被记成 `root_player`，这样 rollout 结束后，我们统一从根节点玩家的视角解释胜负。


In [ ]:
def run_mcts(root_state, num_simulations=200, c=1.4):
    root = MCTSNode(state=root_state.clone())
    root_player = root_state.current_player

    for _ in range(num_simulations):
        node = root

        # 1. Selection
        while node.is_fully_expanded() and node.children and not node.state.is_terminal():
            node = select_child(node, c=c)

        # 2. Expansion
        if not node.state.is_terminal() and not node.is_fully_expanded():
            node = expand(node)

        # 3. Simulation
        value = rollout(node.state, root_player=root_player)

        # 4. Backpropagation
        backpropagate(node, value)

    return root


## 9. 看看根节点学到了什么

做完很多次模拟之后，根节点下每个动作都会有：
- 访问次数
- 平均价值

MCTS 最常见的落子方式之一就是：

**在根节点选择访问次数最多的动作。**


In [ ]:
root_state = TicTacToe()
root = run_mcts(root_state, num_simulations=500)

print('初始棋盘：')
print(root_state.render())
print()
print('根节点各动作统计：')
for action, child in sorted(root.children.items()):
    print(f'action={action}, visits={child.visits}, q={child.q_value:.3f}')

best_action = max(root.children.items(), key=lambda kv: kv[1].visits)[0]
print()
print('MCTS 推荐动作:', best_action)


## 10. 用 MCTS 自己下一步看看

下面我们让 MCTS 真正下一手，你会看到它通常会偏向中心或明显更强的位置。


In [ ]:
game = TicTacToe()
print('落子前：')
print(game.render())

root = run_mcts(game, num_simulations=500)
best_action = max(root.children.items(), key=lambda kv: kv[1].visits)[0]
game.step(best_action)

print()
print('MCTS 选择动作:', best_action)
print('落子后：')
print(game.render())


## 11. MCTS 和 AlphaGo Zero 的关系

你现在可以先把这节课和 AlphaGo Zero 做一个连接：

在这节课里：
- rollout 是随机模拟
- 扩展后没有神经网络帮忙

但在 AlphaGo Zero 里：
- 神经网络提供策略先验 `policy`
- 神经网络提供局面价值 `value`
- MCTS 不再纯靠随机 rollout，而是被神经网络引导

所以你可以把 AlphaGo Zero 理解成：

**神经网络增强版的 MCTS。**


## 12. 这节课最该记住什么

如果你想抓住 MCTS 的主线，就记住：

- 它不是把整棵树暴力搜完
- 而是通过反复模拟，把搜索预算集中到更值得看的分支上

并且每轮都在重复这四步：
- 选择
- 扩展
- 模拟
- 回传


## 13. 下一课最自然学什么

学完这节后，最自然的下一步就是：
- `MCTS + policy/value network` 是怎么结合的
- 简化版 AlphaZero 训练数据 `(state, pi, z)` 是怎么来的

也就是说，下一课最适合做：

**Mini AlphaZero 的核心训练循环。**
